# QALB beat classifier: data, results and confusion matrix

This notebook walks through the MIT-BIH data, the inter-patient split, the trained model and the held-out DS2 results. Run `python train.py` first so `reports/metrics.json` and the model exist.

In [ ]:
import sys, json
sys.path.insert(0, '..')
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import matplotlib.pyplot as plt
from qalb_ml.config import CLASSES, DS1, DS2, FS, PRE, POST, REPORT_DIR
from qalb_ml.data import load_record, list_records
from qalb_ml.preprocess import preprocess_window
from qalb_ml.dataset import load_ds1, load_ds2
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False})

## 1. Data
44 usable records remain after dropping the four paced ones (102, 104, 107, 217). Only lead MLII is used.

In [ ]:
print(len(list_records()), 'records;', len(DS1), 'in DS1,', len(DS2), 'in DS2')
sig, fs, peaks, labels = load_record('100')
t = np.arange(10 * fs) / fs
plt.figure(figsize=(10, 2.8))
plt.plot(t, sig[:10 * fs], lw=0.8)
m = peaks < 10 * fs
plt.plot(peaks[m] / fs, sig[peaks[m]], 'r.', label='annotated beats')
plt.xlabel('time (s)'); plt.ylabel('mV'); plt.title('Record 100, first 10 s'); plt.legend(); plt.show()

## 2. Preprocessing
Band-pass 0.5-40 Hz plus baseline removal, then 0.25 s before and 0.4 s after each R peak.

In [ ]:
out = preprocess_window(sig[:10 * fs], fs)
fig, ax = plt.subplots(2, 1, figsize=(10, 4.5), sharex=True)
ax[0].plot(t, sig[:10 * fs], lw=0.8); ax[0].set_title('raw')
ax[1].plot(t[:len(out['clean'])], out['clean'], lw=0.8)
ax[1].plot(out['peaks'] / FS, out['clean'][out['peaks']], 'r.'); ax[1].set_title('cleaned, detected R peaks')
plt.tight_layout(); plt.show()

## 3. Class balance and the inter-patient split
DS1 trains and DS2 tests. No patient appears in both.

In [ ]:
X1, y1, g1 = load_ds1(); X2, y2, g2 = load_ds2()
import pandas as pd
print(pd.DataFrame({'DS1': np.bincount(y1, minlength=5), 'DS2': np.bincount(y2, minlength=5)}, index=CLASSES))
print('features per beat:', X1.shape[1])

In [ ]:
from IPython.display import Image
Image(str(REPORT_DIR / 'class_distribution.png'))

## 4. Beat shapes per class
Mean normalised beat per class in DS1 (the raw samples are the last block of features).

In [ ]:
from qalb_ml.features import feature_dim, morphology_raw
n_raw = (PRE + POST + 2) // 3
fig, ax = plt.subplots(1, 4, figsize=(12, 2.6), sharey=True)
for k, c in enumerate(['N', 'S', 'V', 'F']):
    rows = X1[y1 == CLASSES.index(c)][:, -n_raw:]
    ax[k].plot(rows.mean(axis=0)); ax[k].set_title(f'{c} (n={len(rows)})')
plt.tight_layout(); plt.show()

## 5. Model comparison
Models are chosen with patient-grouped cross-validation on DS1 only. DS2 was scored once.

In [ ]:
res = json.loads((REPORT_DIR / 'metrics.json').read_text())
cv = pd.DataFrame(res['cross_validation_ds1']).T
print('selected model:', res['selected_model'])
cv

In [ ]:
rows = []
for name, m in res['test_ds2'].items():
    rows.append({'model': name, 'accuracy': m['accuracy'], 'macro F1 (N,S,V,F)': m['macro_f1_scored'],
                 'V sensitivity': m['v_sensitivity'], 'S sensitivity': m['s_sensitivity']})
pd.DataFrame(rows).set_index('model').round(3)

In [ ]:
Image(str(REPORT_DIR / 'f1_by_class.png'))

## 6. Confusion matrix of the selected model on DS2

In [ ]:
best = res['test_ds2'][res['selected_model']]
print(pd.DataFrame(best['per_class']).T.round(3))
Image(str(REPORT_DIR / 'confusion_best.png'))

In [ ]:
Image(str(REPORT_DIR / 'confusion_best_normalised.png'))

The Q class has only 8 training and 7 test beats once paced records are removed, so it is reported but excluded from the macro average. Most errors are S beats read as N and F beats read as N or V, the usual weak spots for inter-patient evaluation.

## 7. Alerts on a test record
The model and rule layer applied to 60 s of DS2 record 233, which has runs of ventricular beats.

In [ ]:
sys.path.insert(0, '..')
from predict import predict_window
sig, fs, peaks, labels = load_record('233')
res_w = predict_window(sig[fs*300:fs*360], fs)
print(res_w['rhythm'])
for a in res_w['alerts']: print(a['severity'], a['type'], a['confidence'], a['message'])
print(res_w['disclaimer'])

In [ ]:
t = np.arange(len(sig[fs*300:fs*360])) / fs
plt.figure(figsize=(11, 3))
plt.plot(t, sig[fs*300:fs*360], lw=0.7, color='0.3')
colors = {'N': 'tab:blue', 'S': 'tab:orange', 'V': 'tab:red', 'F': 'tab:purple', 'Q': 'gray'}
for b in res_w['beats']:
    plt.axvline(b['time_s'], color=colors[b['class']], alpha=0.35, lw=1)
plt.title('Predicted beat classes (blue N, orange S, red V, purple F)'); plt.xlabel('s'); plt.xlim(0, 20); plt.show()